# Epic Clarity Episode Hydration

### Derived Table
This notebook derives disease episodes from existing OMOP condition_occurrence data.

### Strategy:
- Identify patients with chronic conditions (hypertension, diabetes, hyperlipidemia, etc.)
- Create a disease episode for each patient's first occurrence of each condition
- episode_concept_id = 'Disease Episode' concept
- episode_type_concept_id = 32817 (EHR)

### Chronic Conditions Included:
- 320128: Essential hypertension
- 432867: Hyperlipidemia  
- 201826: Type 2 diabetes mellitus
- 442077: Anxiety disorder
- 140673: Hypothyroidism
- 45768910: Uncomplicated asthma
- 433736: Obesity
- 442588: Obstructive sleep apnea syndrome

### Dependencies:
- `_exponent.omop_epic.condition_occurrence` must be populated with mapped concepts

In [ ]:
%sql
-- Create episode candidates from multiple chronic conditions
CREATE OR REPLACE TEMP VIEW episode_candidate AS
SELECT
  co.person_id,
  co.condition_concept_id AS episode_object_concept_id,
  MIN(co.condition_start_date) AS episode_start_date,
  CAST(MIN(co.condition_start_date) AS TIMESTAMP) AS episode_start_datetime,
  CAST(NULL AS DATE) AS episode_end_date,
  CAST(NULL AS TIMESTAMP) AS episode_end_datetime,
  CAST(NULL AS BIGINT) AS episode_parent_id,
  CAST(1 AS INT) AS episode_number,
  CONCAT_WS(
    chr(31),
    'epic_clarity',
    'episode',
    CAST(co.condition_concept_id AS STRING),
    CAST(co.person_id AS STRING),
    CAST(MIN(co.condition_start_date) AS STRING)
  ) AS episode_source_value,
  'epic_clarity' AS source_system
FROM _exponent.omop_epic.condition_occurrence co
WHERE co.condition_concept_id IN (
  320128,   -- Essential hypertension
  432867,   -- Hyperlipidemia
  201826,   -- Type 2 diabetes mellitus
  442077,   -- Anxiety disorder
  140673,   -- Hypothyroidism
  45768910, -- Uncomplicated asthma
  433736,   -- Obesity
  442588    -- Obstructive sleep apnea syndrome
)
GROUP BY co.person_id, co.condition_concept_id

In [ ]:
# %sql
# -- Preview candidates
# SELECT * FROM episode_candidate LIMIT 10

In [ ]:
%sql
-- Insert into episode table
INSERT INTO _exponent.omop_epic.episode (
  person_id,
  episode_concept_id,
  episode_start_date,
  episode_start_datetime,
  episode_end_date,
  episode_end_datetime,
  episode_parent_id,
  episode_number,
  episode_object_concept_id,
  episode_type_concept_id,
  episode_source_value,
  episode_source_concept_id
)
SELECT
  ec.person_id,
  disease_episode.concept_id AS episode_concept_id,
  ec.episode_start_date,
  ec.episode_start_datetime,
  ec.episode_end_date,
  ec.episode_end_datetime,
  ec.episode_parent_id,
  ec.episode_number,
  ec.episode_object_concept_id,
  32817 AS episode_type_concept_id,   -- EHR record
  ec.episode_source_value,
  0 AS episode_source_concept_id
FROM episode_candidate ec
CROSS JOIN (
  SELECT concept_id
  FROM _exponent.omop.concept
  WHERE LOWER(concept_name) = 'disease episode'
    AND invalid_reason IS NULL
  LIMIT 1
) disease_episode
WHERE NOT EXISTS (
  SELECT 1 FROM _exponent.omop_epic.episode e
  WHERE e.episode_source_value = ec.episode_source_value
)

In [ ]:
%sql
-- Validation: episode counts by condition
SELECT 
  e.episode_object_concept_id,
  c.concept_name,
  COUNT(*) AS episode_count
FROM _exponent.omop_epic.episode e
JOIN _exponent.omop.concept c ON c.concept_id = e.episode_object_concept_id
GROUP BY e.episode_object_concept_id, c.concept_name
ORDER BY episode_count DESC